# AquaCrisis: exact 10,000-replicate bootstrap CIs for Table 3 LLMs

This notebook reproduces the **LLM point estimates in Table 3** from the exact 1,500-post prediction artifacts and then computes:

1. **95% percentile stratified-bootstrap CIs** for every LLM macro-F1 cell in Table 3;
2. **paired native-vs-translated differences** using the same sampled post indices;
3. **paired five-shot-vs-zero-shot differences** using the same sampled post indices.

### Exact input artifacts

Place these two files in the **same folder as this notebook**:

- `aquacrisis_three_llms_gold1500_results (1).zip`
  - exact 1,500-post predictions for **Gemma 4, GPT-4.1-mini, and Qwen 3.5**
  - this notebook reads `combined/all_predictions_long.csv` directly from the ZIP
- `predictions_long (1).csv`
  - exact 1,500-post predictions for all four **GPT-4o-mini** conditions

Do **not** substitute the later `llm_outputs` files or `batch_predictions_parsed(2).csv`; those are different runs and do not reproduce Table 3.

### Bootstrap procedure

For each task separately, posts are resampled **with replacement within gold-label strata**.
All systems use the **same bootstrap sample indices within a task**, so paired differences preserve post-level dependence.
Macro-F1 is computed over the predefined gold labels only; invalid/out-of-schema predictions remain in the sample and count as incorrect rather than being dropped.

- Replicates: **10,000**
- Confidence level: **95%**
- Interval: **percentile**
- Random seed: **42**

In [ ]:
from pathlib import Path
import zipfile
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

SEED = 42
N_BOOT = 10_000
CI_LEVEL = 0.95
BLOCK_SIZE = 250

BASE_DIR = Path.cwd()

THREE_LLM_ZIP = BASE_DIR / "aquacrisis_three_llms_gold1500_results (1).zip"
GPT4O_PREDICTIONS = BASE_DIR / "predictions_long (1).csv"

OUT_DIR = BASE_DIR / "bootstrap_outputs"
OUT_DIR.mkdir(exist_ok=True)

print("Working directory:", BASE_DIR.resolve())
print("Three-LLM ZIP exists:", THREE_LLM_ZIP.exists())
print("GPT-4o prediction file exists:", GPT4O_PREDICTIONS.exists())

if not THREE_LLM_ZIP.exists():
    raise FileNotFoundError(THREE_LLM_ZIP)
if not GPT4O_PREDICTIONS.exists():
    raise FileNotFoundError(GPT4O_PREDICTIONS)

## 1. Label definitions

In [ ]:
TASK_A_LABELS = [
    "Water quality / safety / public health",
    "Service disruption / repair / infrastructure",
    "Water conservation / drought / demand management",
    "Flood / stormwater / wastewater / sewer",
    "Environmental sustainability / waste / recycling / biodiversity",
    "Public education / heritage / community engagement",
    "Routine / institutional / customer service / other",
]

TASK_B_LABELS = [
    "Alert / warning",
    "Instruction / advice to public",
    "Operational update / resolution",
    "Reassurance / safety information",
    "Education / awareness",
    "Institutional promotion / community news",
    "Other / unclear",
]

TASK_B1_MAP = {
    "Alert / warning": "Risk / incident communication",
    "Instruction / advice to public": "Risk / incident communication",
    "Operational update / resolution": "Risk / incident communication",
    "Reassurance / safety information": "Risk / incident communication",
    "Education / awareness": "Education / awareness",
    "Institutional promotion / community news":
        "Institutional / community communication",
    "Other / unclear": "Other / unclear",
}

TASK_B1_LABELS = [
    "Risk / incident communication",
    "Education / awareness",
    "Institutional / community communication",
    "Other / unclear",
]

TASKS = {
    "Task A": ("task_a_gold", "task_a_pred", TASK_A_LABELS, None),
    "Task B": ("task_b_gold", "task_b_pred", TASK_B_LABELS, None),
    "Task B.1": ("task_b_gold", "task_b_pred", TASK_B1_LABELS, TASK_B1_MAP),
}

MODEL_ORDER = ["Gemma 4", "GPT-4.1-mini", "GPT-4o-mini", "Qwen 3.5"]

## 2. Load the exact prediction artifacts

In [ ]:
with zipfile.ZipFile(THREE_LLM_ZIP) as z:
    member = "combined/all_predictions_long.csv"
    if member not in z.namelist():
        raise FileNotFoundError(
            f"{member} not found inside {THREE_LLM_ZIP.name}"
        )
    with z.open(member) as f:
        three = pd.read_csv(f, dtype={"post_id": str})

gpt4o = pd.read_csv(GPT4O_PREDICTIONS, dtype={"post_id": str})

print("Three-LLM rows:", len(three))
print("Three-LLM experiments:", three["experiment"].nunique())
print("GPT-4o rows:", len(gpt4o))
print("GPT-4o experiments:", gpt4o["experiment"].nunique())

In [ ]:
three_small = three[
    [
        "experiment", "model", "text_setting", "shot", "post_id",
        "task_a_gold", "task_b_gold", "task_a_pred", "task_b_pred",
        "invalid",
    ]
].copy()

gpt4o_small = gpt4o[
    [
        "experiment", "text_setting", "shot", "post_id",
        "task_a_gold", "task_b_gold", "task_a_pred", "task_b_pred",
        "invalid",
    ]
].copy()

gpt4o_small["model"] = "GPT-4o-mini"

all_predictions = pd.concat(
    [three_small, gpt4o_small],
    ignore_index=True,
)

counts = all_predictions.groupby("experiment").size()
unique_ids = all_predictions.groupby("experiment")["post_id"].nunique()

assert len(counts) == 16, f"Expected 16 LLM conditions, found {len(counts)}"
assert (counts == 1500).all(), counts[counts != 1500]
assert (unique_ids == 1500).all(), unique_ids[unique_ids != 1500]

experiment_meta = (
    all_predictions[
        ["experiment", "model", "text_setting", "shot"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

def experiment_sort_key(row):
    return (
        MODEL_ORDER.index(row["model"]),
        0 if row["shot"] == "zero" else 1,
        0 if row["text_setting"] == "native" else 1,
    )

experiment_meta["_sort"] = experiment_meta.apply(
    experiment_sort_key, axis=1
)
experiment_meta = experiment_meta.sort_values("_sort").drop(
    columns="_sort"
).reset_index(drop=True)

EXPERIMENT_ORDER = experiment_meta["experiment"].tolist()

display(experiment_meta)

## 3. Confirm the shared 1,500-post gold set

In [ ]:
reference_exp = EXPERIMENT_ORDER[0]

reference = (
    all_predictions[
        all_predictions["experiment"] == reference_exp
    ][["post_id", "task_a_gold", "task_b_gold"]]
    .drop_duplicates("post_id")
    .set_index("post_id")
    .sort_index()
)

assert len(reference) == 1500

for exp in EXPERIMENT_ORDER[1:]:
    current = (
        all_predictions[
            all_predictions["experiment"] == exp
        ][["post_id", "task_a_gold", "task_b_gold"]]
        .drop_duplicates("post_id")
        .set_index("post_id")
        .sort_index()
    )

    assert current.index.equals(reference.index), f"Post IDs differ for {exp}"
    assert current["task_a_gold"].equals(reference["task_a_gold"]), (
        f"Task A gold labels differ for {exp}"
    )
    assert current["task_b_gold"].equals(reference["task_b_gold"]), (
        f"Task B gold labels differ for {exp}"
    )

POST_IDS = reference.index.tolist()

print("Validated common gold set:", len(POST_IDS), "posts")
print("\nTask A gold counts:")
print(reference["task_a_gold"].value_counts())
print("\nTask B gold counts:")
print(reference["task_b_gold"].value_counts())

## 4. Reproduce Table 3 point estimates

In [ ]:
def fixed_label_macro_f1(y_true, y_pred, labels):
    # Macro-F1 over predefined gold labels only.
    # Missing/out-of-schema predictions are retained as an invalid sentinel,
    # so they count as incorrect rather than being dropped.
    y_true = pd.Series(y_true).astype(str)
    y_pred = pd.Series(y_pred, dtype="object")

    y_pred = y_pred.where(
        y_pred.isin(labels),
        "__INVALID__",
    )

    return f1_score(
        y_true,
        y_pred,
        labels=labels,
        average="macro",
        zero_division=0,
    )


def score_experiment(df):
    a = fixed_label_macro_f1(
        df["task_a_gold"],
        df["task_a_pred"],
        TASK_A_LABELS,
    )

    b = fixed_label_macro_f1(
        df["task_b_gold"],
        df["task_b_pred"],
        TASK_B_LABELS,
    )

    gold_b1 = df["task_b_gold"].map(TASK_B1_MAP)
    pred_b1 = df["task_b_pred"].map(TASK_B1_MAP)

    b1 = fixed_label_macro_f1(
        gold_b1,
        pred_b1,
        TASK_B1_LABELS,
    )

    return a, b, b1


point_rows = []

for exp in EXPERIMENT_ORDER:
    d = all_predictions[
        all_predictions["experiment"] == exp
    ].copy()

    a, b, b1 = score_experiment(d)

    meta = experiment_meta[
        experiment_meta["experiment"] == exp
    ].iloc[0]

    point_rows.append({
        "experiment": exp,
        "model": meta["model"],
        "text": meta["text_setting"],
        "shot": meta["shot"],
        "Task A": a,
        "Task B": b,
        "Task B.1": b1,
        "invalid_rows_percent":
            100 * d["invalid"].fillna(True).astype(bool).mean(),
    })

point_estimates = pd.DataFrame(point_rows)

display(point_estimates)

## 5. Fail fast if the files do not match Table 3

In [ ]:
TABLE3 = {
    "gemma4_zero_translated": (0.658, 0.352, 0.563, 0.00),
    "gemma4_five_translated": (0.624, 0.331, 0.553, 12.27),
    "gemma4_zero_native":     (0.633, 0.346, 0.552, 0.00),
    "gemma4_five_native":     (0.566, 0.292, 0.497, 19.80),

    "gpt41mini_zero_translated": (0.638, 0.359, 0.494, 0.00),
    "gpt41mini_five_translated": (0.617, 0.325, 0.466, 0.00),
    "gpt41mini_zero_native":     (0.616, 0.328, 0.491, 0.07),
    "gpt41mini_five_native":     (0.596, 0.315, 0.463, 0.00),

    "gpt4o_mini_zero_translated": (0.620, 0.392, 0.646, 0.00),
    "gpt4o_mini_five_translated": (0.624, 0.381, 0.660, 0.00),
    "gpt4o_mini_zero_native":     (0.601, 0.384, 0.623, 0.00),
    "gpt4o_mini_five_native":     (0.605, 0.372, 0.650, 0.00),

    "qwen35_zero_translated": (0.567, 0.277, 0.472, 0.13),
    "qwen35_five_translated": (0.584, 0.269, 0.441, 0.13),
    "qwen35_zero_native":     (0.558, 0.277, 0.461, 0.53),
    "qwen35_five_native":     (0.573, 0.242, 0.430, 1.20),
}

check_rows = []

for _, r in point_estimates.iterrows():
    expected = TABLE3[r["experiment"]]

    for value, target, name in [
        (r["Task A"], expected[0], "Task A"),
        (r["Task B"], expected[1], "Task B"),
        (r["Task B.1"], expected[2], "Task B.1"),
    ]:
        assert abs(value - target) < 0.0005, (
            f"{r['experiment']} {name}: "
            f"{value:.6f} does not reproduce Table 3 {target:.3f}"
        )

    assert abs(r["invalid_rows_percent"] - expected[3]) < 0.01, (
        f"{r['experiment']} invalid-rate mismatch"
    )

    check_rows.append({
        "experiment": r["experiment"],
        "computed_A": r["Task A"],
        "table3_A": expected[0],
        "computed_B": r["Task B"],
        "table3_B": expected[1],
        "computed_B1": r["Task B.1"],
        "table3_B1": expected[2],
        "computed_invalid_percent": r["invalid_rows_percent"],
        "table3_invalid_percent": expected[3],
    })

point_check = pd.DataFrame(check_rows)
point_check.to_csv(
    OUT_DIR / "llm_point_estimate_check.csv",
    index=False,
)

print("PASS: all 16 LLM conditions reproduce Table 3.")
display(point_check)

## 6. Stratified paired-bootstrap implementation

For each task separately, rows are sampled with replacement inside each gold-label stratum. The same sampled row indices are used for all 16 systems, which preserves pairing for translated-vs-native and five-vs-zero comparisons.

In [ ]:
def build_task_arrays(task_name):
    gold_col, pred_col, labels, group_map = TASKS[task_name]

    label_to_int = {
        label: i for i, label in enumerate(labels)
    }
    invalid_code = len(labels)

    base = (
        all_predictions[
            all_predictions["experiment"] == reference_exp
        ][["post_id", gold_col]]
        .drop_duplicates("post_id")
        .set_index("post_id")
        .loc[POST_IDS]
    )

    gold = base[gold_col].copy()
    if group_map is not None:
        gold = gold.map(group_map)

    y = np.array(
        [label_to_int[x] for x in gold],
        dtype=np.int16,
    )

    pred_matrix = []

    for exp in EXPERIMENT_ORDER:
        d = (
            all_predictions[
                all_predictions["experiment"] == exp
            ][["post_id", pred_col]]
            .drop_duplicates("post_id")
            .set_index("post_id")
            .loc[POST_IDS]
        )

        pred = d[pred_col].copy()

        if group_map is not None:
            pred = pred.map(group_map)

        codes = np.array(
            [
                label_to_int.get(x, invalid_code)
                for x in pred
            ],
            dtype=np.int16,
        )

        pred_matrix.append(codes)

    return y, np.stack(pred_matrix, axis=0)


def macro_f1_from_confusion(conf):
    K = conf.shape[-2]

    tp = np.stack(
        [conf[..., c, c] for c in range(K)],
        axis=-1,
    )

    pred_counts = conf[..., :, :K].sum(axis=-2)
    gold_counts = conf.sum(axis=-1)

    fp = pred_counts - tp
    fn = gold_counts - tp

    denom = 2 * tp + fp + fn

    class_f1 = np.divide(
        2 * tp,
        denom,
        out=np.zeros_like(tp, dtype=float),
        where=denom != 0,
    )

    return class_f1.mean(axis=-1)


def point_scores_from_codes(y, pred_matrix, K):
    scores = []

    for s in range(pred_matrix.shape[0]):
        code_values = y * (K + 1) + pred_matrix[s]

        conf = np.bincount(
            code_values,
            minlength=K * (K + 1),
        ).reshape(K, K + 1)

        scores.append(
            macro_f1_from_confusion(conf)
        )

    return np.asarray(scores)


def stratified_bootstrap_scores(
    y,
    pred_matrix,
    labels,
    n_boot=N_BOOT,
    seed=SEED,
    block_size=BLOCK_SIZE,
):
    K = len(labels)
    S = pred_matrix.shape[0]
    N = len(y)

    strata = [
        np.flatnonzero(y == c)
        for c in range(K)
    ]

    counts = [len(x) for x in strata]

    gold_template = np.concatenate(
        [
            np.full(n, c, dtype=np.int16)
            for c, n in enumerate(counts)
        ]
    )

    assert len(gold_template) == N

    rng = np.random.default_rng(seed)

    bootstrap = np.empty(
        (S, n_boot),
        dtype=np.float64,
    )

    start = 0

    while start < n_boot:
        b = min(block_size, n_boot - start)

        sampled_parts = [
            rng.choice(
                indices,
                size=(b, len(indices)),
                replace=True,
            )
            for indices in strata
        ]

        sampled_indices = np.concatenate(
            sampled_parts,
            axis=1,
        )

        sampled_pred = pred_matrix[:, sampled_indices]

        cell_code = (
            gold_template[None, None, :] * (K + 1)
            + sampled_pred
        )

        offsets = (
            (
                np.arange(S)[:, None] * b
                + np.arange(b)[None, :]
            )
            * (K * (K + 1))
        )[:, :, None]

        encoded = (cell_code + offsets).ravel()

        counts_flat = np.bincount(
            encoded,
            minlength=S * b * K * (K + 1),
        )

        conf = counts_flat.reshape(
            S, b, K, K + 1
        )

        bootstrap[:, start:start + b] = (
            macro_f1_from_confusion(conf)
        )

        start += b

    point = point_scores_from_codes(
        y,
        pred_matrix,
        K,
    )

    return point, bootstrap

## 7. Run 10,000 replicates

In [ ]:
bootstrap_by_task = {}

for task_name, (_, _, labels, _) in TASKS.items():
    print("Running:", task_name)

    y, pred_matrix = build_task_arrays(task_name)

    point, boot = stratified_bootstrap_scores(
        y,
        pred_matrix,
        labels,
        n_boot=N_BOOT,
        seed=SEED,
        block_size=BLOCK_SIZE,
    )

    bootstrap_by_task[task_name] = {
        "point": point,
        "boot": boot,
    }

print("Done.")

## 8. Individual 95% CIs for all 48 LLM cells

In [ ]:
ci_rows = []

meta_by_exp = (
    experiment_meta
    .set_index("experiment")
    .loc[EXPERIMENT_ORDER]
)

alpha = 1 - CI_LEVEL
q_low = alpha / 2
q_high = 1 - alpha / 2

for task_name, result in bootstrap_by_task.items():
    point = result["point"]
    boot = result["boot"]

    lower = np.quantile(
        boot,
        q_low,
        axis=1,
    )
    upper = np.quantile(
        boot,
        q_high,
        axis=1,
    )

    for i, exp in enumerate(EXPERIMENT_ORDER):
        meta = meta_by_exp.loc[exp]

        ci_rows.append({
            "experiment": exp,
            "model": meta["model"],
            "text": meta["text_setting"],
            "shot": meta["shot"],
            "task": task_name,
            "macro_f1": point[i],
            "ci_low": lower[i],
            "ci_high": upper[i],
            "n_boot": N_BOOT,
            "seed": SEED,
        })

individual_ci = pd.DataFrame(ci_rows)

individual_ci.to_csv(
    OUT_DIR / "llm_macro_f1_bootstrap_ci.csv",
    index=False,
)

individual_ci["estimate_ci"] = individual_ci.apply(
    lambda r:
        f"{r['macro_f1']:.3f} "
        f"[{r['ci_low']:.3f}, {r['ci_high']:.3f}]",
    axis=1,
)

individual_table = (
    individual_ci
    .pivot_table(
        index=["model", "text", "shot"],
        columns="task",
        values="estimate_ci",
        aggfunc="first",
    )
    .reset_index()
)

display(individual_table)

## 9. Paired native-vs-translated differences

In [ ]:
exp_index = {
    exp: i
    for i, exp in enumerate(EXPERIMENT_ORDER)
}

translation_rows = []

for model in MODEL_ORDER:
    for shot in ["zero", "five"]:

        subset = experiment_meta[
            (experiment_meta["model"] == model)
            & (experiment_meta["shot"] == shot)
        ]

        native_exp = subset[
            subset["text_setting"] == "native"
        ]["experiment"].iloc[0]

        translated_exp = subset[
            subset["text_setting"] == "translated"
        ]["experiment"].iloc[0]

        ni = exp_index[native_exp]
        ti = exp_index[translated_exp]

        for task_name, result in bootstrap_by_task.items():
            point = result["point"]
            boot = result["boot"]

            delta = point[ti] - point[ni]
            delta_boot = boot[ti] - boot[ni]

            low, high = np.quantile(
                delta_boot,
                [q_low, q_high],
            )

            translation_rows.append({
                "model": model,
                "shot": shot,
                "task": task_name,
                "native_experiment": native_exp,
                "translated_experiment": translated_exp,
                "delta_translated_minus_native": delta,
                "ci_low": low,
                "ci_high": high,
                "ci_excludes_zero": bool(
                    (low > 0) or (high < 0)
                ),
                "n_boot": N_BOOT,
                "seed": SEED,
            })

translation_ci = pd.DataFrame(translation_rows)

translation_ci.to_csv(
    OUT_DIR / "llm_translation_paired_bootstrap_ci.csv",
    index=False,
)

display(translation_ci)

## 10. Paired five-shot-vs-zero-shot differences

In [ ]:
fewshot_rows = []

for model in MODEL_ORDER:
    for text in ["native", "translated"]:

        subset = experiment_meta[
            (experiment_meta["model"] == model)
            & (experiment_meta["text_setting"] == text)
        ]

        zero_exp = subset[
            subset["shot"] == "zero"
        ]["experiment"].iloc[0]

        five_exp = subset[
            subset["shot"] == "five"
        ]["experiment"].iloc[0]

        zi = exp_index[zero_exp]
        fi = exp_index[five_exp]

        for task_name, result in bootstrap_by_task.items():
            point = result["point"]
            boot = result["boot"]

            delta = point[fi] - point[zi]
            delta_boot = boot[fi] - boot[zi]

            low, high = np.quantile(
                delta_boot,
                [q_low, q_high],
            )

            fewshot_rows.append({
                "model": model,
                "text": text,
                "task": task_name,
                "zero_experiment": zero_exp,
                "five_experiment": five_exp,
                "delta_five_minus_zero": delta,
                "ci_low": low,
                "ci_high": high,
                "ci_excludes_zero": bool(
                    (low > 0) or (high < 0)
                ),
                "n_boot": N_BOOT,
                "seed": SEED,
            })

fewshot_ci = pd.DataFrame(fewshot_rows)

fewshot_ci.to_csv(
    OUT_DIR / "llm_fewshot_paired_bootstrap_ci.csv",
    index=False,
)

display(fewshot_ci)

## 11. Export a compact appendix LaTeX table

In [ ]:
row_order = []

for model in MODEL_ORDER:
    for text in ["translated", "native"]:
        for shot in ["zero", "five"]:
            row_order.append((model, text, shot))

wide = (
    individual_ci
    .pivot_table(
        index=["model", "text", "shot"],
        columns="task",
        values=["macro_f1", "ci_low", "ci_high"],
        aggfunc="first",
    )
)

def fmt_ci(row, task):
    p = row[("macro_f1", task)]
    lo = row[("ci_low", task)]
    hi = row[("ci_high", task)]
    return f"{p:.3f} [{lo:.3f}, {hi:.3f}]"

latex_lines = [
    r"\begin{table*}[t]",
    r"\centering",
    r"\small",
    r"\begin{tabular}{lllccc}",
    r"\toprule",
    r"Model & Text & Setting & Task A & Task B & Task B.1 \\",
    r"\midrule",
]

for model, text, shot in row_order:
    row = wide.loc[(model, text, shot)]

    setting = "zero-shot" if shot == "zero" else "five-shot"

    latex_lines.append(
        f"{model} & {text} & {setting} & "
        f"{fmt_ci(row, 'Task A')} & "
        f"{fmt_ci(row, 'Task B')} & "
        f"{fmt_ci(row, 'Task B.1')} \\\\"
    )

latex_lines.extend([
    r"\bottomrule",
    r"\end{tabular}",
    r"\caption{LLM macro-F1 with 95\% percentile confidence intervals from 10,000 stratified-bootstrap replicates. Resampling is performed within gold-label strata. Invalid predictions are retained and scored as incorrect.}",
    r"\label{tab:llm-bootstrap-ci}",
    r"\end{table*}",
])

latex_text = "\n".join(latex_lines)

latex_path = OUT_DIR / "llm_macro_f1_bootstrap_ci_table.tex"
latex_path.write_text(latex_text, encoding="utf-8")

print(latex_text)
print("\nSaved:", latex_path)

## 12. Paper-ready paired translation summary

In [ ]:
translation_display = translation_ci.copy()

translation_display["delta"] = translation_display[
    "delta_translated_minus_native"
].map(lambda x: f"{x:+.3f}")

translation_display["95% CI"] = translation_display.apply(
    lambda r:
        f"[{r['ci_low']:+.3f}, {r['ci_high']:+.3f}]",
    axis=1,
)

display(
    translation_display[
        [
            "model", "shot", "task",
            "delta", "95% CI",
            "ci_excludes_zero",
        ]
    ]
)

print("\nPaired translation differences whose 95% CI excludes zero:\n")

display(
    translation_display[
        translation_display["ci_excludes_zero"]
    ][
        [
            "model", "shot", "task",
            "delta", "95% CI",
        ]
    ]
)

## Interpretation note

A percentile CI for a paired difference that **includes zero** should not be described as clear evidence of a difference under this bootstrap analysis.

Use the paired-difference intervals for native-vs-translated or five-vs-zero claims. Do not use overlap or non-overlap of the individual-system intervals as a significance test.